<a href="https://colab.research.google.com/github/Shivani02082004/azure-ad-access-review/blob/main/%20Access%20review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
border_color = '#cccccc'

"""
Azure AD-style Access Review Simulator
----------------------------------------
Reads a user access CSV and flags accounts that would typically be caught
in a real IAM access review:

  1. STALE ACCOUNTS      - no login in 90+ days
  2. OVER-PRIVILEGED      - holds a high-privilege admin role
  3. HIGH RISK            - admin role + no MFA enabled
  4. STALE + PRIVILEGED   - the worst combination: unused admin access

Usage:
    python3 access_review.py users.csv
"""

import csv
import sys
from datetime import datetime

HIGH_PRIV_ROLES = {"Global Admin", "User Admin", "Application Admin", "Billing Admin"}
STALE_THRESHOLD_DAYS = 90


def load_users(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def days_since(date_str):
    last_login = datetime.strptime(date_str, "%Y-%m-%d")
    return (datetime.now() - last_login).days


def review(users):
    findings = []
    for u in users:
        flags = []
        idle_days = days_since(u["last_login_date"])
        is_privileged = u["role"] in HIGH_PRIV_ROLES
        has_mfa = u["mfa_enabled"].strip().lower() == "true"

        if idle_days >= STALE_THRESHOLD_DAYS:
            flags.append(f"STALE (no login in {idle_days} days)")

        if is_privileged:
            flags.append(f"OVER-PRIVILEGED ({u['role']})")

        if is_privileged and not has_mfa:
            flags.append("HIGH RISK: privileged role without MFA")

        if flags:
            findings.append({
                "user_id": u["user_id"],
                "name": u["full_name"],
                "department": u["department"],
                "role": u["role"],
                "flags": flags
            })
    return findings


def print_report(findings, total_users):
    print("=" * 60)
    print("AZURE AD ACCESS REVIEW REPORT")
    print("=" * 60)
    print(f"Total accounts reviewed: {total_users}")
    print(f"Accounts flagged: {len(findings)}\n")

    for f in findings:
        print(f"[{f['user_id']}] {f['name']} — {f['department']} — {f['role']}")
        for flag in f["flags"]:
            print(f"    -> {flag}")
        print()

    if not findings:
        print("No issues found. All accounts pass review.")


if __name__ == "__main__":
    # In a Colab/Jupyter environment, sys.argv[1] might be '-f'
    # which is an internal IPython/Jupyter argument.
    # We want to use 'users.csv' by default if no actual file path is provided.
    if len(sys.argv) > 1 and not sys.argv[1].startswith('-'):
        path = sys.argv[1]
    else:
        path = "users.csv"
    users = load_users(path)
    findings = review(users)
    print_report(findings, len(users))

AZURE AD ACCESS REVIEW REPORT
Total accounts reviewed: 20
Accounts flagged: 15

[U1001] Aarav Shah — Claims — Application Admin
    -> OVER-PRIVILEGED (Application Admin)

[U1005] Karan Verma — Customer Service — Application Admin
    -> STALE (no login in 200 days)
    -> OVER-PRIVILEGED (Application Admin)
    -> HIGH RISK: privileged role without MFA

[U1007] Vikram Joshi — IT — Global Admin
    -> OVER-PRIVILEGED (Global Admin)

[U1008] Divya Pillai — Claims — Standard User
    -> STALE (no login in 200 days)

[U1009] Arjun Kapoor — IT — Billing Admin
    -> STALE (no login in 95 days)
    -> OVER-PRIVILEGED (Billing Admin)
    -> HIGH RISK: privileged role without MFA

[U1010] Meera Menon — Claims — Security Reader
    -> STALE (no login in 200 days)

[U1011] Sai Reddy — Customer Service — Billing Admin
    -> STALE (no login in 120 days)
    -> OVER-PRIVILEGED (Billing Admin)

[U1012] Nikhil Desai — Claims — Application Admin
    -> OVER-PRIVILEGED (Application Admin)

[U1013] Po